# Q1


In [37]:

import numpy as np
import pandas as pd



rng = np.random.default_rng(0)

price_paths = np.array([
    [42.67, 45.53, 47.07, 47.56, 47.80, 48.43, 46.93, 46.57],
    [46.35, 43.15, 42.51, 40.51, 41.50, 41.00, 39.16, 41.11],
    [43.17, 45.16, 45.37, 44.30, 45.35, 47.23, 47.35, 46.30],
    [45.24, 45.67, 46.18, 46.22, 45.69, 44.24, 43.77, 43.57],
    [47.68, 46.32, 46.14, 41.53, 44.84, 45.17, 44.92, 46.09],
    [47.83, 44.70, 43.05, 43.77, 42.61, 44.32, 44.16, 45.29],
    [45.11, 43.67, 43.14, 44.78, 43.12, 42.36, 41.60, 40.83],
    [46.78, 44.98, 44.53, 45.42, 46.43, 47.67, 47.68, 49.03],
    [43.16, 44.57, 45.99, 47.38, 45.51, 46.27, 46.02, 45.09],
    [46.57, 45.01, 46.73, 46.08, 47.40, 49.14, 49.03, 48.74]
], dtype=float)

p0 = 40.0
n_param_samples = 1000
min_price, max_price = float(price_paths.min()), float(price_paths.max())

In [38]:

#Policies
def sell_first_below(path_prices: np.ndarray, theta_low: float) -> float:
    for price in path_prices:
        if price < theta_low:
            return price
    return path_prices[-1]

def sell_outside_band(path_prices: np.ndarray, theta_low: float, theta_high: float) -> float:
    for price in path_prices:
        if (price < theta_low) or (price > theta_high):
            return price
    return path_prices[-1]

def sell_track(path_prices: np.ndarray, alpha: float, theta_track: float, p0_ref: float) -> float:
    m = p0_ref
    for price in path_prices:
        m = (1 - alpha) * m + alpha * price
        if price >= m + theta_track:
            return price
    return path_prices[-1]

In [39]:
def find_best_sell_low(paths: np.ndarray, n_samples: int = n_param_samples):
    # RANDOM search over theta_low in [min_price, max_price]
    theta_samples = rng.uniform(min_price, max_price, size=n_samples)
    best_mean_price, best_theta_low = -1e18, None
    for theta_low in theta_samples:
        sale_prices = [sell_first_below(path, theta_low) for path in paths]
        mean_price = float(np.mean(sale_prices))
        if mean_price > best_mean_price:
            best_mean_price, best_theta_low = mean_price, float(theta_low)
    return {
        "policy": "sell-low",
        "theta_low": best_theta_low,
        "theta_high": None,
        "alpha": None,
        "theta_track": None,
        "avg_sale_price": best_mean_price,
    }


def find_best_high_low(paths: np.ndarray, n_samples: int = n_param_samples):
    # RANDOM pairs in [min_price, max_price]^2, enforce hi > lo by sorting
    theta_pairs = rng.uniform(min_price, max_price, size=(n_samples, 2))
    theta_pairs.sort(axis=1)

    best_mean_price, best_theta_low, best_theta_high = -1e18, None, None
    for theta_low, theta_high in theta_pairs:
        if theta_high <= theta_low:
            continue
        sale_prices = [sell_outside_band(path, theta_low, theta_high) for path in paths]
        mean_price = float(np.mean(sale_prices))
        if mean_price > best_mean_price:
            best_mean_price = mean_price
            best_theta_low  = float(theta_low)
            best_theta_high = float(theta_high)
    return {
        "policy": "high-low",
        "theta_low": best_theta_low,
        "theta_high": best_theta_high,
        "alpha": None,
        "theta_track": None,
        "avg_sale_price": best_mean_price,
    }


def find_best_track(
    paths: np.ndarray,
    n_samples: int = n_param_samples,
    alpha_min: float = 0.05,
    alpha_max: float = 0.30,
    theta_min: float = 0.0,
    theta_max: float = 5.0,
    p0_ref: float = p0,
):
    # RANDOM search: sample (alpha, theta_track) independently
    alpha_samples = rng.uniform(alpha_min, alpha_max, size=n_samples)
    theta_samples = rng.uniform(theta_min, theta_max, size=n_samples)

    best_mean_price, best_alpha, best_theta_track = -1e18, None, None
    for alpha, theta_track in zip(alpha_samples, theta_samples):
        sale_prices = [sell_track(path, alpha, theta_track, p0_ref) for path in paths]
        mean_price = float(np.mean(sale_prices))
        if mean_price > best_mean_price:
            best_mean_price   = mean_price
            best_alpha        = float(alpha)
            best_theta_track  = float(theta_track)
    return {
        "policy": "track",
        "theta_low": None,
        "theta_high": None,
        "alpha": best_alpha,
        "theta_track": best_theta_track,
        "avg_sale_price": best_mean_price,
    }


In [40]:
result_sell_low = find_best_sell_low(price_paths, n_param_samples)
result_high_low = find_best_high_low(price_paths, n_param_samples)
result_track    = find_best_track(price_paths, n_param_samples)

results = pd.DataFrame([result_sell_low, result_high_low, result_track]) \
            .sort_values("avg_sale_price", ascending=False) \
            .reset_index(drop=True)
results["avg_sale_price"] = results["avg_sale_price"].round(6)
print(results.to_string(index=False))

  policy  theta_low  theta_high    alpha  theta_track  avg_sale_price
   track        NaN         NaN 0.187367     4.135177          46.631
high-low  42.608314   46.063791      NaN          NaN          46.543
sell-low  48.269301         NaN      NaN          NaN          45.456


# Q2

In [41]:

def simulate_paths(p0: float, sigma: float, t: int, n: int, seed=None) -> np.ndarray:
    """
    Returns an (n, t) matrix with column0 = p0, and subsequent columns built by
    cumulative Gaussian increments with std = sigma.
    """
    rng_local = np.random.default_rng(seed)
    if t <= 0:
        return np.empty((n, 0))
    if t == 1:
        return np.full((n, 1), p0, dtype=float)

    eps = rng_local.normal(0.0, sigma, size=(n, t-1))
    return np.concatenate([np.full((n, 1), p0), p0 + np.cumsum(eps, axis=1)], axis=1)

# regenerate Q2 paths with p0 as the first column (exactly 40)
prices_q2 = simulate_paths(p0=40.0, sigma=2.0, t=8, n=5000, seed=0)
min_price, max_price = float(prices_q2.min()), float(prices_q2.max())

prices_q2


array([[40.        , 40.25146044, 39.98725072, ..., 40.4065575 , 41.12974761, 43.7377477 ],
       [40.        , 41.89416193, 40.48669145, ..., 36.79195155, 32.14189   , 31.70430667],
       [40.        , 37.50817811, 36.0436434 , ..., 35.14578619, 37.23081293, 36.9737436 ],
       ...,
       [40.        , 41.95363654, 42.65136311, ..., 40.42783272, 37.79678721, 37.67800231],
       [40.        , 39.55281712, 41.60708087, ..., 44.77541579, 44.44912863, 44.09340393],
       [40.        , 41.32400317, 40.86974261, ..., 42.90310405, 41.72740652, 41.20742322]])

In [42]:
res_sell_low_q2 = find_best_sell_low(prices_q2, n_param_samples)
res_high_low_q2 = find_best_high_low(prices_q2, n_param_samples)
res_track_q2    = find_best_track(prices_q2, n_param_samples)

summary_q2 = pd.DataFrame([res_sell_low_q2, res_high_low_q2, res_track_q2]) \
                .sort_values("avg_sale_price", ascending=False) \
                .reset_index(drop=True)

print(summary_q2)

     policy  theta_low  theta_high    alpha  theta_track  avg_sale_price
0     track        NaN         NaN  0.16231     2.944708       40.123078
1  high-low  27.425570   44.085035      NaN          NaN       40.115788
2  sell-low  27.107759         NaN      NaN          NaN       40.076212


# Q3

In [43]:

def simulate_paths_ar3(p0: float, sigma: float, t: int, n: int,
                       theta=(0.5, 0.3, 0.2), seed=None) -> np.ndarray:
  
    rng_local = np.random.default_rng(seed)
    eps = rng_local.normal(0.0, sigma, size=(n, t))
    th0, th1, th2 = theta

    prices = np.empty((n, t), dtype=float)

    prices[:, 0] = p0
    if t >= 2:
        prices[:, 1] = th0 * prices[:, 0] + th1 * p0 + th2 * p0 + eps[:, 1] 
    if t >= 3:
        prices[:, 2] = th0 * prices[:, 1] + th1 * prices[:, 0] + th2 * p0 + eps[:, 2]

    for s in range(3, t):
        prices[:, s] = (th0 * prices[:, s-1] +
                        th1 * prices[:, s-2] +
                        th2 * prices[:, s-3] + eps[:, s])
    return prices


# generate sample
prices_q3 = simulate_paths_ar3(p0=40.0, sigma=2.0, t=8, n=5000, theta=(0.5,0.3,0.2), seed=0)
prices_q3


array([[40.        , 39.73579027, 41.14874044, ..., 40.95085806, 43.09627919, 43.64213798],
       [40.        , 37.46915706, 37.4880296 , ..., 35.0421896 , 32.57992138, 31.96286   ],
       [40.        , 39.36739969, 40.50696092, ..., 43.90036847, 41.30239904, 42.69290269],
       ...,
       [40.        , 35.25293391, 37.6252721 , ..., 38.27970568, 38.09543772, 34.26751985],
       [40.        , 38.07583205, 38.24132402, ..., 38.98365006, 36.36932715, 35.36078116],
       [40.        , 37.1844294 , 37.74447559, ..., 38.85352829, 38.44956826, 36.35841271]])

In [44]:

# Ensure Q3 searches use Q3 bounds (not the ones from Q2)
min_price, max_price = float(prices_q3.min()), float(prices_q3.max())


res_sell_low_q3 = find_best_sell_low(prices_q3, n_param_samples)
res_high_low_q3 = find_best_high_low(prices_q3, n_param_samples)
res_track_q3    = find_best_track(prices_q3, n_param_samples)


summary_q3 = pd.DataFrame([
    {"policy": "sell-low", "theta_low": res_sell_low_q3["theta_low"], "theta_high": None,
     "alpha": None, "theta_track": None, "avg_sale_price": res_sell_low_q3["avg_sale_price"]},
    {"policy": "high-low", "theta_low": res_high_low_q3["theta_low"], "theta_high": res_high_low_q3["theta_high"],
     "alpha": None, "theta_track": None, "avg_sale_price": res_high_low_q3["avg_sale_price"]},
    {"policy": "track",    "theta_low": None, "theta_high": None,
     "alpha": res_track_q3["alpha"], "theta_track": res_track_q3["theta_track"],
     "avg_sale_price": res_track_q3["avg_sale_price"]},
]).sort_values("avg_sale_price", ascending=False).reset_index(drop=True)

summary_q3["avg_sale_price"] = summary_q3["avg_sale_price"].round(6)
print(summary_q3)




     policy  theta_low  theta_high     alpha  theta_track  avg_sale_price
0     track        NaN         NaN  0.297896     0.896433       40.878566
1  high-low  28.410295    41.31589       NaN          NaN       40.746442
2  sell-low  28.810289         NaN       NaN          NaN       40.009073


# CI Q2 vs Q3

In [ ]:

SEED = 0
import numpy as np
import pandas as pd

Z95 = 1.959963984540054 
TCRIT_Q1 = 2.262       

def ci_mean(x, z=Z95):
    x = np.asarray(x, dtype=float)
    m  = float(np.mean(x))
    se = float(np.std(x, ddof=1)) / np.sqrt(len(x))
    return m, (m - z*se, m + z*se)

def ci_paired_diff(a, b, z=Z95):
    d = np.asarray(a, float) - np.asarray(b, float)
    return ci_mean(d, z=z)

def samples_for_policy(TABLE, idxs, policy_name, params, p0_ref=40.0):
    n = len(idxs); vals = np.empty(n, dtype=float)
    for k, i in enumerate(idxs):
        prices = TABLE.iloc[i].to_numpy(float) if isinstance(TABLE, pd.DataFrame) else TABLE[i].astype(float, copy=False)
        if policy_name == "track":
            vals[k] = sell_track(prices, params["alpha"], params["theta_track"], p0_ref)
        elif policy_name == "high-low":
            vals[k] = sell_outside_band(prices, params["theta_low"], params["theta_high"])
        elif policy_name == "sell-low":
            vals[k] = sell_first_below(prices, params["theta_low"])
        else:
            raise ValueError(f"unknown policy: {policy_name}")
    return vals

def ci_block_from_params(TABLE, params_by_policy, title, use_all_paths=True, N=5000, seed=SEED, p0_ref=40.0, z=Z95):
    if use_all_paths:
        idxs = np.arange(TABLE.shape[0])
    else:
        rng = np.random.default_rng(seed)
        idxs = rng.integers(0, TABLE.shape[0], size=N)


    s_track   = samples_for_policy(TABLE, idxs, "track",    params_by_policy["track"],    p0_ref=p0_ref)
    s_highlow = samples_for_policy(TABLE, idxs, "high-low", params_by_policy["high-low"], p0_ref=p0_ref)
    s_selllow = samples_for_policy(TABLE, idxs, "sell-low", params_by_policy["sell-low"], p0_ref=p0_ref)

    # Per-policy CIs
    m_tr, ci_tr = ci_mean(s_track, z=z)
    m_hl, ci_hl = ci_mean(s_highlow, z=z)
    m_sl, ci_sl = ci_mean(s_selllow, z=z)

    summary = pd.DataFrame([
        {"policy": "track",    "avg_sale_price": m_tr, "CI95": ci_tr, "params": params_by_policy["track"]},
        {"policy": "high-low", "avg_sale_price": m_hl, "CI95": ci_hl, "params": params_by_policy["high-low"]},
        {"policy": "sell-low", "avg_sale_price": m_sl, "CI95": ci_sl, "params": params_by_policy["sell-low"]},
    ]).sort_values("avg_sale_price", ascending=False).reset_index(drop=True)

    print(f"\n== {title} ==")
    disp = summary.copy()
    disp["avg_sale_price"] = disp["avg_sale_price"].round(4)
    disp["CI95"] = disp["CI95"].apply(lambda t: (round(t[0],4), round(t[1],4)))
    print(disp.to_string(index=False))


    winner = summary.iloc[0]["policy"]
    print("\nPaired differences (winner vs others), 95% CI:")
    if winner == "track":
        d1, ci1 = ci_paired_diff(s_track, s_highlow, z=z); print(f"track - high-low: Δ={d1:.4f}, CI=({ci1[0]:.4f},{ci1[1]:.4f})")
        d2, ci2 = ci_paired_diff(s_track, s_selllow, z=z); print(f"track - sell-low: Δ={d2:.4f}, CI=({ci2[0]:.4f},{ci2[1]:.4f})")
    elif winner == "high-low":
        d1, ci1 = ci_paired_diff(s_highlow, s_track, z=z); print(f"high-low - track: Δ={d1:.4f}, CI=({ci1[0]:.4f},{ci1[1]:.4f})")
        d2, ci2 = ci_paired_diff(s_highlow, s_selllow, z=z); print(f"high-low - sell-low: Δ={d2:.4f}, CI=({ci2[0]:.4f},{ci2[1]:.4f})")
    else:
        d1, ci1 = ci_paired_diff(s_selllow, s_track, z=z); print(f"sell-low - track: Δ={d1:.4f}, CI=({ci1[0]:.4f},{ci1[1]:.4f})")
        d2, ci2 = ci_paired_diff(s_selllow, s_highlow, z=z); print(f"sell-low - high-low: Δ={d2:.4f}, CI=({ci2[0]:.4f},{ci2[1]:.4f})")

    return summary

# --- Build params from the actual tuning dicts (EXACT match) ---
# Q1 (your variable names from tuning: result_* )
params_q1 = {
    "track":    {"alpha": float(result_track["alpha"]),          "theta_track": float(result_track["theta_track"])},
    "high-low": {"theta_low": float(result_high_low["theta_low"]), "theta_high": float(result_high_low["theta_high"])},
    "sell-low": {"theta_low": float(result_sell_low["theta_low"])},
}
# Q2
params_q2 = {
    "track":    {"alpha": float(res_track_q2["alpha"]),          "theta_track": float(res_track_q2["theta_track"])},
    "high-low": {"theta_low": float(res_high_low_q2["theta_low"]), "theta_high": float(res_high_low_q2["theta_high"])},
    "sell-low": {"theta_low": float(res_sell_low_q2["theta_low"])},
}
# Q3
params_q3 = {
    "track":    {"alpha": float(res_track_q3["alpha"]),          "theta_track": float(res_track_q3["theta_track"])},
    "high-low": {"theta_low": float(res_high_low_q3["theta_low"]), "theta_high": float(res_high_low_q3["theta_high"])},
    "sell-low": {"theta_low": float(res_sell_low_q3["theta_low"])},
}


_ = ci_block_from_params(price_paths, params_q1,
                         "Q1 (Given 10 paths) — CIs per strategy at tuned params",
                         use_all_paths=True, seed=SEED, p0_ref=p0, z=TCRIT_Q1)


_ = ci_block_from_params(prices_q2, params_q2,
                         "Q2 (Random-Walk) — CIs per strategy at tuned params",
                         use_all_paths=True, seed=SEED, p0_ref=p0, z=Z95)


_ = ci_block_from_params(prices_q3, params_q3,
                         "Q3 (AR(3)) — CIs per strategy at tuned params",
                         use_all_paths=True, seed=SEED, p0_ref=p0, z=Z95)



== Q1 (Given 10 paths) — CIs per strategy at tuned params ==
  policy  avg_sale_price               CI95                                                            params
   track          46.631   (45.967, 47.295)  {'alpha': 0.18736652310789165, 'theta_track': 4.135176527690557}
high-low          46.543 (45.4215, 47.6645) {'theta_low': 42.60831440589951, 'theta_high': 46.06379090824345}
sell-low          45.456 (44.0904, 46.8216)                                  {'theta_low': 48.26930066123166}

Paired differences (winner vs others), 95% CI:
track - high-low: Δ=0.0880, CI=(-0.6365,0.8125)
track - sell-low: Δ=1.1750, CI=(-0.1980,2.5480)

== Q2 (Random-Walk) — CIs per strategy at tuned params ==
  policy  avg_sale_price               CI95                                                            params
   track         40.1231  (39.9891, 40.257) {'alpha': 0.16231022935938982, 'theta_track': 2.9447079566868357}
high-low         40.1158 (39.9808, 40.2508) {'theta_low': 27.42557044272848